# SGLang and RadixAttention: Sharing the KV Cache Across Requests

The full code of [SGLang and RadixAttention: Sharing the KV Cache Across Requests](https://residual-stream.lvtan71.workers.dev/posts/sglang-radixattention-explained/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/sglang-radixattention-explained) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/sglang-radixattention-explained.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 2 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q torch transformers huggingface_hub

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "sglang-radixattention-explained"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

## `hits.py`

Reuse on multi-turn ShareGPT conversations, where each turn's prompt is the whole conversation so far, at five cache sizes.

In [ ]:
!{sys.executable} hits.py

The post printed:

```text
2161 requests from 429 multi-turn conversations; 1,583,998 prompt tokens, 733 per request
cache size (tokens)    radix tree, any prefix    16-token blocks
            16,384           30.3%                   30.0%
            65,536           80.5%                   79.4%
           262,144           92.0%                   90.9%
         1,048,576           92.0%                   90.9%
         unlimited           92.0%                   90.9%
```

## `schedule.py`

Cache-aware scheduling on SGLang's shared-prefix benchmark design: arrival order, longest-prefix-match first, and longest-prefix-match as SGLang ships it.

In [ ]:
!{sys.executable} schedule.py

The post printed:

```text
                   prompt tokens reused       first 128 arrivals served after this much prefill work
policy             seed 0  seed 1  seed 2     median (seed 0)     slowest (seed 0)
fcfs                 7.9%    8.9%    8.6%       137,343 tokens      262,140 tokens
lpm                 69.9%   57.8%   64.4%       216,775 tokens      507,556 tokens
lpm, as shipped     10.3%   11.7%   11.3%       137,343 tokens      262,140 tokens
```

## `prefill.py`

Time to the first token of a 2,176-token prompt, computed whole and with its 2,048-token prefix cached.

**Note:** GPU; time it with nothing else running.

In [ ]:
!{sys.executable} prefill.py

The post printed:

```text
time to first token, 2,176-token prompt: 2101.5 ms computed whole, 152.6 ms with the 2,048-token prefix cached (13.8x faster)
largest difference in the first token's logits: 0.0e+00; same first token: True
```

## `jump.py`

Constrained JSON decoding, plain and with jump-forward: forward passes and time.

**Note:** GPU.

In [ ]:
!{sys.executable} jump.py

The post printed:

```text
'{"title": "Paged Attention", "year": 2026, "summary": "This blog post discusses the concept of paged attention, which is a strategy for managing information overload by breaking down large amounts of data into smaller, more manageable chunks, allowing users to focus on specific aspects"}'
plain       : 317 output tokens in 312 forward passes, 16.3 s
jump-forward: 317 output tokens in 247 forward passes, 12.0 s
outputs identical to plain decoding: 4 of 5; first differing token: [None, 50, None, None, None]
```